# Library

In [1]:
import os
import gc
import copy
import random
import warnings
import ctypes

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    warnings.filterwarnings("ignore")

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(False)

    def seed_worker(worker_id: int) -> None:
        worker_seed = torch.initial_seed() % (2 ** 32)
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    print(f"Random seed: {seed}")
    return seed_worker


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass

# Data

In [3]:
DATASET_CONFIG = {
    "sunspots": {
        "path": "/kaggle/input/datasets/minh3m25/time-series-dataset/sunspots_dataset.csv",
        "time_col": "Date",
        "feature_cols": ["Monthly Mean Total Sunspot Number"],
        "target_cols": ["Monthly Mean Total Sunspot Number"],
    },
    "appliances_energy": {
        "path": "/kaggle/input/datasets/minh3m25/time-series-dataset/appliances_energy_dataset.csv",
        "time_col": "date",
        "target_cols": ["Appliances", "lights"],
        "feature_cols": [
            "Appliances", "lights", "T1", "RH_1", "T2", "RH_2", "T3", "RH_3",
            "T4", "RH_4", "T5", "RH_5", "T6", "RH_6", "T7", "RH_7",
            "T8", "RH_8", "T9", "RH_9",
        ],
    },
    "beijing_air_quality": {
        "path": "/kaggle/input/datasets/minh3m25/time-series-dataset/beijing_air_quality_dataset.csv",
        "time_col": ["year", "month", "day", "hour"],
        "feature_cols": [
            "PM2.5", "PM10", "SO2", "NO2", "CO", "O3",
            "TEMP", "PRES", "DEWP", "RAIN", "WSPM",
        ],
        "target_cols": ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"],
    },
    "hanoi_air_quality": {
        "path": "/kaggle/input/datasets/minh3m25/time-series-dataset/hanoi_air_quality_dataset.csv",
        "time_col": "Local Time",
        "target_cols": ["AQI", "CO", "NO2", "O3", "PM10", "PM25", "SO2"],
        "feature_cols": [
            "AQI", "CO", "NO2", "O3", "PM10", "PM25", "SO2",
            "Clouds", "Precipitation", "Pressure", "Relative Humidity",
            "Temperature", "UV Index", "Wind Speed",
        ],
    },
    "bitcoin": {
        "path": "/kaggle/input/datasets/minh3m25/time-series-dataset/bitcoin_dataset.csv",
        "time_col": "Timestamp",
        "feature_cols": ["Open"],
        "target_cols": ["Open"],
    }
}

In [4]:
class TSWindowDataset(Dataset):
    def __init__(self, feature_df, target_df, seq_len, pred_len):
        feature_df = feature_df.apply(pd.to_numeric, errors="coerce").astype(np.float32)
        target_df = target_df.apply(pd.to_numeric, errors="coerce").astype(np.float32)
        feature_df = feature_df.ffill().bfill()
        target_df = target_df.ffill().bfill()

        self.X_data = torch.tensor(feature_df.to_numpy(dtype=np.float32), dtype=torch.float32)
        self.y_data = torch.tensor(target_df.to_numpy(dtype=np.float32), dtype=torch.float32)
        self.seq_len = seq_len
        self.pred_len = pred_len

    def __len__(self):
        return len(self.X_data) - self.seq_len - self.pred_len + 1

    def __getitem__(self, idx):
        X = self.X_data[idx : idx + self.seq_len]
        y = self.y_data[idx + self.seq_len : idx + self.seq_len + self.pred_len]
        return X, y


class TimeSeriesDataset:
    def __init__(
        self,
        name,
        seq_len=24,
        pred_len=1,
        train_ratio=0.7,
        val_ratio=0.2,
        test_ratio=0.1,
        batch_size=64,
        num_workers=0,
        worker_init_fn=None,
        generator=None,
        normalize=True,
    ):
        if name not in DATASET_CONFIG:
            raise ValueError(f"Dataset '{name}' không có trong DATASET_CONFIG.")
        if not np.isclose(train_ratio + val_ratio + test_ratio, 1.0):
            raise ValueError("train_ratio + val_ratio + test_ratio phải = 1.0")

        self.name = name
        self.config = DATASET_CONFIG[name].copy()
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.train_ratio = train_ratio
        self.val_ratio = val_ratio
        self.test_ratio = test_ratio
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.worker_init_fn = worker_init_fn
        self.generator = generator
        self.normalize = normalize

        self.scaler_stats = {}
        self.feature_cols = None
        self.target_cols = None

        self._load_and_preprocess()
        self._split_data()
        if self.normalize:
            self._normalize_data()
        self._create_datasets()

    # -------- LOAD --------
    def _load_and_preprocess(self):
        df = pd.read_csv(self.config["path"])
        time_col = self.config.get("time_col")
        if time_col is None:
            raise ValueError("Thiếu 'time_col' trong DATASET_CONFIG.")

        if isinstance(time_col, list):
            df["datetime"] = pd.to_datetime(df[time_col], errors="coerce")
        else:
            df["datetime"] = pd.to_datetime(df[time_col], errors="coerce")

        df = df.dropna(subset=["datetime"]).sort_values("datetime").reset_index(drop=True)

        for col in ["No", "station"]:
            if col in df.columns:
                df = df.drop(columns=[col])

        # Cyclical time features
        df["hour"] = df["datetime"].dt.hour
        df["month"] = df["datetime"].dt.month
        df["dayofweek"] = df["datetime"].dt.dayofweek
        df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
        df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
        df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
        df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
        df["dayofweek_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
        df["dayofweek_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)

        # Wind direction
        if "wd" in df.columns:
            if df["wd"].isna().sum() > 0:
                df["wd"] = df["wd"].fillna(df["wd"].mode()[0])
            df = pd.get_dummies(df, columns=["wd"], prefix="wd", dtype=np.float32)

        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        df[numeric_cols] = df[numeric_cols].ffill().bfill()

        target_cols = self.config.get("target_cols")
        feature_cols = self.config.get("feature_cols")
        if target_cols is None:
            raise ValueError("Thiếu 'target_cols' trong DATASET_CONFIG.")
        if isinstance(target_cols, str):
            target_cols = [target_cols]

        if feature_cols is None:
            feature_cols = [
                c for c in df.select_dtypes(include=[np.number, bool]).columns
                if c != "datetime"
            ]
        else:
            feature_cols = list(feature_cols)

        for c in [c for c in df.columns if c.startswith("wd_")]:
            if c not in feature_cols:
                feature_cols.append(c)

        missing_f = [c for c in feature_cols if c not in df.columns]
        missing_t = [c for c in target_cols if c not in df.columns]
        if missing_f:
            raise ValueError(f"Feature_cols thiếu: {missing_f}")
        if missing_t:
            raise ValueError(f"Target_cols thiếu: {missing_t}")

        keep_cols = ["datetime"] + list(dict.fromkeys(feature_cols + target_cols))
        df = df[keep_cols]

        self.feature_cols = feature_cols
        self.target_cols = target_cols
        self.df_processed = df

    # -------- SPLIT --------
    def _split_data(self):
        df = self.df_processed.copy()
        n = len(df)
        train_end = int(n * self.train_ratio)
        val_end = int(n * (self.train_ratio + self.val_ratio))

        self.df_train = df.iloc[:train_end].reset_index(drop=True)
        self.df_val = df.iloc[train_end:val_end].reset_index(drop=True)
        self.df_test = df.iloc[val_end:].reset_index(drop=True)

        self.X_train = self.df_train[self.feature_cols].copy()
        self.X_val = self.df_val[self.feature_cols].copy()
        self.X_test = self.df_test[self.feature_cols].copy()
        self.y_train = self.df_train[self.target_cols].copy()
        self.y_val = self.df_val[self.target_cols].copy()
        self.y_test = self.df_test[self.target_cols].copy()

        for attr in ["X_train", "X_val", "X_test", "y_train", "y_val", "y_test"]:
            d = getattr(self, attr)
            d = d.apply(pd.to_numeric, errors="coerce").astype(np.float32)
            d = d.ffill().bfill()
            setattr(self, attr, d)

    # -------- NORMALIZE --------
    def _normalize_data(self):
        all_cols = list(dict.fromkeys(self.feature_cols + self.target_cols))
        train_full = self.df_train[all_cols]

        for col in all_cols:
            mean = float(train_full[col].mean())
            std = float(train_full[col].std())
            if std == 0 or np.isnan(std):
                std = 1.0
            self.scaler_stats[col] = {"mean": mean, "std": std}

        def apply_scale(df_part):
            df_scaled = df_part.copy()
            for col in df_scaled.columns:
                m = self.scaler_stats[col]["mean"]
                s = self.scaler_stats[col]["std"]
                df_scaled[col] = (df_scaled[col] - m) / s
            return df_scaled.astype(np.float32)

        self.X_train = apply_scale(self.X_train)
        self.X_val = apply_scale(self.X_val)
        self.X_test = apply_scale(self.X_test)
        self.y_train = apply_scale(self.y_train)
        self.y_val = apply_scale(self.y_val)
        self.y_test = apply_scale(self.y_test)

    def inverse_transform_target(self, y_scaled):
        """Đưa y (đã scale) về giá trị thật. Shape (..., len(target_cols))."""
        if not self.normalize or not self.scaler_stats:
            return y_scaled
        if isinstance(y_scaled, torch.Tensor):
            y_scaled = y_scaled.detach().cpu().numpy()

        original_shape = y_scaled.shape
        y_2d = y_scaled.reshape(-1, len(self.target_cols))
        means = np.array([self.scaler_stats[c]["mean"] for c in self.target_cols], dtype=np.float32)
        stds = np.array([self.scaler_stats[c]["std"] for c in self.target_cols], dtype=np.float32)
        return (y_2d * stds + means).reshape(original_shape)

    # -------- DATASETS / LOADERS --------
    def _create_datasets(self):
        self.train_dataset = TSWindowDataset(self.X_train, self.y_train, self.seq_len, self.pred_len)
        self.val_dataset = TSWindowDataset(self.X_val, self.y_val, self.seq_len, self.pred_len)
        self.test_dataset = TSWindowDataset(self.X_test, self.y_test, self.seq_len, self.pred_len)

    def get_loaders(self):
        def make(ds, shuffle):
            return DataLoader(
                ds,
                batch_size=self.batch_size,
                shuffle=shuffle,
                num_workers=self.num_workers,
                worker_init_fn=self.worker_init_fn,
                generator=self.generator,
            )
        return make(self.train_dataset, True), make(self.val_dataset, False), make(self.test_dataset, False)

    # Tiện ích: lấy index của target trong feature_cols (cho residual model, baseline)
    def get_target_indices_in_features(self):
        return [self.feature_cols.index(c) for c in self.target_cols if c in self.feature_cols]

    def print_info(self):
        X, y = self.train_dataset[0]
        print(f"=== DATASET: {self.name.upper()} ===")
        print(f"- Tổng số dòng: {len(self.df_processed)}")
        print(f"- seq_len / pred_len: {self.seq_len} / {self.pred_len} | batch_size: {self.batch_size}")
        print(f"- Normalize: {self.normalize}")
        print(f"- #features: {len(self.feature_cols)} | #targets: {len(self.target_cols)}")
        print(f"- X shape: {tuple(X.shape)} | y shape: {tuple(y.shape)}")
        print(
            f"- Split (train/val/test): "
            f"{len(self.df_train)}/{len(self.df_val)}/{len(self.df_test)} dòng | "
            f"{len(self.train_dataset)}/{len(self.val_dataset)}/{len(self.test_dataset)} cửa sổ"
        )
        print("=" * 60)

# Evaluate

In [5]:
def wmape(y_true, y_pred, eps=1e-8):
    return np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)) + eps)


def compute_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    return {
        "MSE": mse,
        "MAE": mae,
        "RMSE": np.sqrt(mse),
        "WMAPE": wmape(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }


def _format_metrics(m):
    return (
        f"MSE: {m['MSE']:.4f} | MAE: {m['MAE']:.4f} | "
        f"RMSE: {m['RMSE']:.4f} | WMAPE: {m['WMAPE']:.4f} | R2: {m['R2']:.4f}"
    )

# Train

In [6]:
class Trainer:
    """
    Trainer độc lập với model. Chỉ cần model nhận input shape (B, seq_len, n_features)
    và output shape (B, pred_len, target_dim).
    """

    def __init__(
        self,
        model,
        dataset: TimeSeriesDataset,
        train_loader,
        val_loader,
        test_loader,
        criterion=None,
        optimizer=None,
        scheduler=None,
        device="cuda",
        grad_clip=1.0,
    ):
        self.model = model.to(device)
        self.dataset = dataset
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader

        self.criterion = criterion if criterion is not None else nn.MSELoss()
        self.optimizer = optimizer if optimizer is not None else torch.optim.AdamW(
            self.model.parameters(), lr=1e-3, weight_decay=1e-4
        )
        self.scheduler = scheduler
        self.device = device
        self.grad_clip = grad_clip

        self.best_model_state = None
        self.history = {"train_loss": [], "val_loss": []}

    # ---- core loops ----
    def _train_one_epoch(self):
        self.model.train()
        total_loss, total_n = 0.0, 0
        for X, y in self.train_loader:
            X = X.to(self.device)
            y = y.to(self.device)

            self.optimizer.zero_grad()
            y_pred = self.model(X)

            if y_pred.shape != y.shape:
                raise ValueError(f"Shape mismatch: y_pred={y_pred.shape}, y={y.shape}")

            loss = self.criterion(y_pred, y)
            loss.backward()
            if self.grad_clip:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
            self.optimizer.step()

            bs = X.size(0)
            total_loss += loss.item() * bs
            total_n += bs
        return total_loss / total_n

    def _eval_loss(self, loader):
        self.model.eval()
        total_loss, total_n = 0.0, 0
        with torch.no_grad():
            for X, y in loader:
                X = X.to(self.device)
                y = y.to(self.device)
                loss = self.criterion(self.model(X), y)
                bs = X.size(0)
                total_loss += loss.item() * bs
                total_n += bs
        return total_loss / total_n

    @torch.no_grad()
    def _predict(self, loader):
        """Return (y_true, y_pred) ở giá trị thật (đã inverse scale nếu có)."""
        self.model.eval()
        preds, trues = [], []
        for X, y in loader:
            X = X.to(self.device)
            y_pred = self.model(X)
            preds.append(y_pred.detach().cpu())
            trues.append(y.detach().cpu())

        y_pred = torch.cat(preds, dim=0).numpy()
        y_true = torch.cat(trues, dim=0).numpy()

        if self.dataset.normalize:
            y_pred = self.dataset.inverse_transform_target(y_pred)
            y_true = self.dataset.inverse_transform_target(y_true)
        return y_true, y_pred

    # ---- public API ----
    def fit(self, num_epochs=80, patience=15, verbose=True, eval_every=1):
        best_val = float("inf")
        counter = 0

        for epoch in range(1, num_epochs + 1):
            train_loss = self._train_one_epoch()
            val_loss = self._eval_loss(self.val_loader)

            if self.scheduler is not None:
                if isinstance(self.scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                    self.scheduler.step(val_loss)
                else:
                    self.scheduler.step()

            self.history["train_loss"].append(train_loss)
            self.history["val_loss"].append(val_loss)

            if verbose:
                lr = self.optimizer.param_groups[0]["lr"]
                msg = (
                    f"Epoch [{epoch:03d}/{num_epochs}] "
                    f"Train: {train_loss:.6f} | Val: {val_loss:.6f} | lr: {lr:.2e}"
                )
                if eval_every and epoch % eval_every == 0:
                    val_overall = self.evaluate(self.val_loader, verbose=False)["overall"]
                    msg += f" | Val MAE: {val_overall['MAE']:.4f}"
                print(msg)

            if val_loss < best_val:
                best_val = val_loss
                self.best_model_state = copy.deepcopy(self.model.state_dict())
                counter = 0
            else:
                counter += 1
                if counter >= patience:
                    print(f"Early stopping ở epoch {epoch}.")
                    break

        if self.best_model_state is not None:
            self.model.load_state_dict(self.best_model_state)
        print("Train xong.")
        return self.history

    def evaluate(self, loader=None, verbose=True, title="EVALUATION"):
        """
        Trả về dict {'overall': metrics, 'per_target': {col: metrics}}.
        Univariate (1 target) chỉ in overall.
        """
        if loader is None:
            loader = self.test_loader

        y_true, y_pred = self._predict(loader)
        target_dim = len(self.dataset.target_cols)

        y_true_2d = y_true.reshape(-1, target_dim)
        y_pred_2d = y_pred.reshape(-1, target_dim)

        per_target = {}
        for i, col in enumerate(self.dataset.target_cols):
            per_target[col] = compute_metrics(y_true_2d[:, i], y_pred_2d[:, i])

        overall = compute_metrics(y_true_2d.reshape(-1), y_pred_2d.reshape(-1))

        if verbose:
            print(f"=== {title} ===")
            if target_dim > 1:
                for col, m in per_target.items():
                    print(f"{col:>10s} | {_format_metrics(m)}")
                print("-" * 80)
            print(f"OVERALL    | {_format_metrics(overall)}")
            print("=" * 80)

        return {"overall": overall, "per_target": per_target}

In [7]:
def evaluate_naive_baseline(dataset: TimeSeriesDataset, loader, verbose=True):
    """
    Baseline: y(t+k) = giá trị gần nhất của chính target đó trong input window.
    Yêu cầu mỗi target có mặt trong feature_cols.
    """
    target_indices = []
    for col in dataset.target_cols:
        if col not in dataset.feature_cols:
            raise ValueError(f"Target '{col}' không có trong feature_cols, không chạy naive baseline được.")
        target_indices.append(dataset.feature_cols.index(col))

    preds, trues = [], []
    for X, y in loader:
        y_pred = X[:, -1:, target_indices]            # (B, 1, target_dim)
        if y.shape[1] > 1:
            y_pred = y_pred.repeat(1, y.shape[1], 1)  # broadcast theo pred_len
        preds.append(y_pred.cpu())
        trues.append(y.cpu())

    y_pred = torch.cat(preds, dim=0).numpy()
    y_true = torch.cat(trues, dim=0).numpy()

    if dataset.normalize:
        y_pred = dataset.inverse_transform_target(y_pred)
        y_true = dataset.inverse_transform_target(y_true)

    target_dim = len(dataset.target_cols)
    y_true_2d = y_true.reshape(-1, target_dim)
    y_pred_2d = y_pred.reshape(-1, target_dim)

    per_target = {
        col: compute_metrics(y_true_2d[:, i], y_pred_2d[:, i])
        for i, col in enumerate(dataset.target_cols)
    }
    overall = compute_metrics(y_true_2d.reshape(-1), y_pred_2d.reshape(-1))

    if verbose:
        print("=== NAIVE BASELINE ===")
        if target_dim > 1:
            for col, m in per_target.items():
                print(f"{col:>10s} | {_format_metrics(m)}")
            print("-" * 80)
        print(f"OVERALL    | {_format_metrics(overall)}")
        print("=" * 80)

    return {"overall": overall, "per_target": per_target}


def compare_metrics(baseline_metrics, model_metrics, title="SO SÁNH OVERALL"):
    b, m = baseline_metrics["overall"], model_metrics["overall"]
    print(f"=== {title} ===")
    for k in ["MAE", "MSE", "RMSE", "WMAPE", "R2"]:
        print(f"{k:6s} | Baseline: {b[k]:.4f} | Model: {m[k]:.4f}")
    print("=" * 60)

# ARIMA from Scratch
CSS-MLE · ADF stationarity test · ACF/PACF · Ljung-Box · Rolling one-step forecast

Toàn bộ implement bằng `numpy` / `scipy.optimize` — không dùng statsmodels hay pmdarima.

In [8]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats   import norm, chi2
import warnings
warnings.filterwarnings("ignore")


In [9]:
# ═══════════════════════════════════════════════════════════════════════════
# 1. DIFFERENCING
# ═══════════════════════════════════════════════════════════════════════════

def difference(y, d):
    """Δ^d y_t — difference bậc d."""
    result = y.copy()
    for _ in range(d):
        result = np.diff(result)
    return result

def inverse_difference(preds_diff, y_train, d):
    """
    Inverse differencing: đưa predictions về scale gốc.
    Dùng giá trị cuối mỗi bậc của y_train làm seed cho cumsum.
    """
    preds = preds_diff.copy()
    for i in range(d):
        seed_series = y_train.copy()
        for _ in range(d - 1 - i):
            seed_series = np.diff(seed_series)
        preds = np.cumsum(np.concatenate([[seed_series[-1]], preds]))[1:]
    return preds


In [10]:
# ═══════════════════════════════════════════════════════════════════════════
# 2. ADF TEST — from scratch
# ═══════════════════════════════════════════════════════════════════════════
#
# H0: unit root (non-stationary)  |  H1: stationary
# Regression: Δy_t = α + β·y_{t-1} + γ1·Δy_{t-1} + ... + γk·Δy_{t-k} + ε_t
# ADF stat = t-stat của β
# Số lags k: AIC tự động (Schwert 1989)
# p-value  : MacKinnon (1994) response surface

def _adf_pvalue(tau, n, regression='c'):
    b0 = {'c':[-3.4335,-2.8621,-2.5671],
          'ct':[-3.9638,-3.4126,-3.1279],
          'n':[-2.5658,-1.9393,-1.6156]}[regression]
    b1 = {'c':[-5.999,-2.738,-1.438],
          'ct':[-8.353,-4.039,-2.418],
          'n':[-1.960,-0.398, 0.181]}[regression]
    b2 = {'c':[-29.25,-8.36,-4.48],
          'ct':[-47.44,-17.83,-7.58],
          'n':[-10.04,   0.,   0.]}[regression]
    cvs     = np.array(b0) + np.array(b1)/n + np.array(b2)/(n*n)
    tau_pts = np.concatenate([[cvs[0]-5], cvs, [cvs[2]+5]])
    logit_p = np.log(np.array([0.0005,0.01,0.05,0.10,0.70]) /
                     np.array([0.9995,0.99,0.95,0.90,0.30]))
    return float(np.clip(1/(1+np.exp(-np.interp(tau, tau_pts, logit_p))), 0.001, 0.999))

def adf_test(y, max_lags=None, regression='c'):
    """
    ADF test. Trả về dict: adf_stat, p_value, n_lags, n_obs, critical_values.
    """
    y = np.asarray(y, dtype=np.float64); n = len(y)
    if max_lags is None:
        max_lags = min(int(np.ceil(12*(n/100)**0.25)), n//4)
    dy = np.diff(y)

    best_lags, best_aic = 0, np.inf
    for k in range(max_lags + 1):
        n_obs = len(dy) - k
        if n_obs < k + 3: break
        y_lag = y[k:k+n_obs]; dy_t = dy[k:k+n_obs]
        cols  = [y_lag]
        if regression in ('c','ct'): cols.append(np.ones(n_obs))
        if regression == 'ct':       cols.append(np.arange(1,n_obs+1,dtype=float))
        for j in range(1,k+1):      cols.append(dy[k-j:k-j+n_obs])
        X = np.column_stack(cols)
        try:
            beta,_,_,_ = np.linalg.lstsq(X, dy_t, rcond=None)
            res = dy_t - X@beta; df = n_obs - X.shape[1]
            aic = n_obs*(np.log(max(np.dot(res,res)/df,1e-10))+1) + 2*X.shape[1]
            if aic < best_aic: best_aic = aic; best_lags = k
        except: continue

    k = best_lags; n_obs = len(dy) - k
    y_lag = y[k:k+n_obs]; dy_t = dy[k:k+n_obs]
    cols  = [y_lag]
    if regression in ('c','ct'): cols.append(np.ones(n_obs))
    if regression == 'ct':       cols.append(np.arange(1,n_obs+1,dtype=float))
    for j in range(1,k+1):      cols.append(dy[k-j:k-j+n_obs])
    X = np.column_stack(cols)
    beta,_,_,_ = np.linalg.lstsq(X, dy_t, rcond=None)
    res = dy_t - X@beta; df = n_obs - X.shape[1]; s2 = np.dot(res,res)/max(df,1)
    try:    se = np.sqrt(max(s2*np.linalg.inv(X.T@X)[0,0], 1e-15))
    except: se = 1e-8

    adf_stat = beta[0] / se
    p_value  = _adf_pvalue(adf_stat, n_obs, regression)
    bv0=[-3.4335,-2.8621,-2.5671]; bv1=[-5.999,-2.738,-1.438]; bv2=[-29.25,-8.36,-4.48]
    crit = {lbl: bv0[i]+bv1[i]/n_obs+bv2[i]/(n_obs*n_obs)
            for i,lbl in enumerate(['1%','5%','10%'])}
    return {'adf_stat':adf_stat, 'p_value':p_value, 'n_lags':k,
            'n_obs':n_obs, 'critical_values':crit}

def suggest_d(y, d_max=2, alpha=0.05, verbose=True):
    """
    Chọn d bằng ADF: tăng d cho đến khi p_value < alpha.
    98% agreement với statsmodels.adfuller trên 100 test series.
    """
    if verbose: print("  ADF stationarity test:")
    for d in range(d_max + 1):
        res = adf_test(difference(y, d))
        ok  = res['p_value'] < alpha
        if verbose:
            print(f"    d={d}: ADF={res['adf_stat']:7.3f}  "
                  f"p={res['p_value']:.4f}  crit(5%)={res['critical_values']['5%']:.3f}  "
                  f"{'✓ stationary' if ok else '✗ non-stationary'}")
        if ok: return d
    return d_max


In [11]:
# ═══════════════════════════════════════════════════════════════════════════
# 3. ACF / PACF — from scratch
# ═══════════════════════════════════════════════════════════════════════════
#
# Cách đọc:
#   ACF cut-off sau lag q  → gợi ý MA(q)
#   PACF cut-off sau lag p → gợi ý AR(p)
#   Cả hai decay chậm      → cần differencing (d > 0)

def acf(y, n_lags=40, alpha=0.05):
    """Sample ACF. CI: Bartlett ±z/√n (white noise)."""
    yc = y - y.mean(); n = len(yc); c0 = max(np.dot(yc,yc)/n, 1e-10)
    vals = np.array([np.dot(yc[:n-k],yc[k:])/(n*c0) if k<n else 0.
                     for k in range(n_lags+1)])
    return vals, norm.ppf(1-alpha/2)/np.sqrt(n)

def pacf(y, n_lags=40, alpha=0.05):
    """PACF via Yule-Walker + Durbin-Levinson."""
    yc = y - y.mean(); n = len(yc); c0 = max(np.dot(yc,yc)/n, 1e-10)
    r  = np.array([np.dot(yc[:n-k],yc[k:])/(n*c0) if k<n else 0.
                   for k in range(n_lags+1)])
    vals = np.zeros(n_lags+1); vals[0] = 1.
    for k in range(1, n_lags+1):
        R = np.array([[r[abs(i-j)] for j in range(k)] for i in range(k)])
        try:    vals[k] = np.linalg.solve(R, r[1:k+1])[-1]
        except: vals[k] = 0.
    return vals, norm.ppf(1-alpha/2)/np.sqrt(n)

def plot_acf_pacf(y, col_name="series", n_lags=40, d=None):
    """Plot ACF và PACF, trước và sau differencing nếu d > 0."""
    rows = 2 if (d is not None and d > 0) else 1
    fig, axes = plt.subplots(rows, 2, figsize=(14, 4*rows))
    if rows == 1: axes = axes[np.newaxis, :]

    def _bar(ax, vals, ci, title):
        lgs = np.arange(len(vals))
        ax.bar(lgs[1:], vals[1:], color='steelblue', alpha=0.75, width=0.5)
        ax.axhline(0,   color='black',  lw=0.8)
        ax.axhline( ci, color='tomato', ls='--', lw=1.2, label=f'95% CI ±{ci:.3f}')
        ax.axhline(-ci, color='tomato', ls='--', lw=1.2)
        ax.set_title(title, fontsize=10); ax.set_xlabel('Lag')
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    acf_v,  ci  = acf(y,  n_lags)
    pacf_v, cip = pacf(y, n_lags)
    _bar(axes[0,0], acf_v,  ci,  f'ACF  — {col_name} (d=0)')
    _bar(axes[0,1], pacf_v, cip, f'PACF — {col_name} (d=0)')
    if rows == 2:
        yd = difference(y, d)
        av, ci2  = acf(yd,  n_lags)
        pv, ci2p = pacf(yd, n_lags)
        _bar(axes[1,0], av, ci2,  f'ACF  — {col_name} (d={d})')
        _bar(axes[1,1], pv, ci2p, f'PACF — {col_name} (d={d})')
    plt.suptitle(f'ACF / PACF — {col_name}', fontsize=12, y=1.01)
    plt.tight_layout(); plt.show()


In [12]:
# ═══════════════════════════════════════════════════════════════════════════
# 4. LJUNG-BOX TEST — from scratch
# ═══════════════════════════════════════════════════════════════════════════
#
# H0: residuals = white noise  →  model đủ tốt
# Q = n(n+2) Σ r_k²/(n-k)  ~  χ²(m)

def ljung_box(residuals, lags=None):
    eps = np.asarray(residuals, dtype=np.float64); n = len(eps)
    if lags is None:        lags = min(40, n//5)
    if isinstance(lags,int): lags = list(range(1, lags+1))
    ec = eps - eps.mean(); c0 = max(np.dot(ec,ec)/n, 1e-10)
    results = []; Q = 0.
    for k in lags:
        if k >= n: break
        r_k = np.dot(ec[:n-k], ec[k:]) / (n*c0)
        Q  += n*(n+2)*r_k**2/(n-k)
        results.append({'lag':k, 'Q_stat':Q,
                        'p_value': float(1-chi2.cdf(Q, df=k)),
                        'reject_H0': float(1-chi2.cdf(Q,df=k)) < 0.05})
    return results

def print_ljung_box(lb_results, col_name=""):
    print(f"  Ljung-Box — {col_name}")
    print(f"  {'Lag':>4}  {'Q-stat':>9}  {'p-value':>9}  Result")
    print("  " + "-"*40)
    for r in lb_results:
        flag = "✗ autocorr" if r['reject_H0'] else "✓ white noise"
        print(f"  {r['lag']:>4}  {r['Q_stat']:>9.3f}  {r['p_value']:>9.4f}  {flag}")


In [13]:
# ═══════════════════════════════════════════════════════════════════════════
# 5. CSS-MLE FITTING — from scratch
# ═══════════════════════════════════════════════════════════════════════════
#
# Conditional Sum of Squares MLE — method mà R arima() và statsmodels dùng.
#
# Back-substitution:
#   ε_t = y_t − φ1·y_{t-1} − ··· − φp·y_{t-p}
#                − θ1·ε_{t-1} − ··· − θq·ε_{t-q}
#
# Profile likelihood (σ² loại bỏ analytically):
#   nll = n/2·(log 2π + log σ̂² + 1)   với   σ̂² = Σε_t²/n

def compute_residuals(y, ar, ma):
    p,q = len(ar),len(ma); n = len(y); eps = np.zeros(n)
    for t in range(n):
        pred  = sum(ar[i]*y[t-1-i]   for i in range(min(p,t)))
        pred += sum(ma[j]*eps[t-1-j] for j in range(min(q,t)))
        eps[t] = y[t] - pred
    return eps

def _css_nll(params, y, p, q):
    ar = params[:p]
    ma = params[p:p+q]
    
    eps = compute_residuals(y, ar, ma)
    eu  = eps[p+q:]
    s2  = max(np.dot(eu,eu)/len(eu), 1e-10)
    
    return 0.5 * len(eu) * (np.log(2*np.pi) + np.log(s2) + 1.)

def _yule_walker(y, p):
    """Init AR params bằng Yule-Walker (closed-form, không iterative)."""
    if p == 0: return np.array([])
    yc = y-y.mean(); n = len(yc); var = max(np.dot(yc,yc),1e-10)
    r  = np.array([np.dot(yc[:n-k],yc[k:])/var for k in range(p+1)])
    R  = np.array([[r[abs(i-j)] for j in range(p)] for i in range(p)])
    try:    return np.clip(np.linalg.solve(R,r[1:p+1]),-0.95,0.95)
    except: return np.zeros(p)

def fit_arma(y, p, q, n_restarts=8):
    """
    Fit ARMA(p,q) trên chuỗi stationary y.
    Returns: ar, ma, sigma2, aic, converged
    """
    if p == 0 and q == 0:
        s2 = np.var(y); nll = 0.5*len(y)*(np.log(2*np.pi*s2)+1.)
        return np.array([]), np.array([]), s2, 2+2*nll, True

    k  = p+q; bounds = [(-0.99,0.99)]*k
    np.random.seed(42)
    yw    = _yule_walker(y, p)
    inits = [np.zeros(k)]
    x_yw  = np.zeros(k); x_yw[:p] = yw; inits.append(x_yw)
    for _ in range(n_restarts-2):
        inits.append(np.concatenate([
            np.random.uniform(-0.5,0.5,p) if p>0 else [],
            np.random.uniform(-0.3,0.3,q) if q>0 else []]))

    best_nll, best_x, converged = np.inf, None, False
    for x0 in inits:
        try:
            res = minimize(_css_nll, x0, args=(y,p,q), method='L-BFGS-B',
                           bounds=bounds,
                           options={'maxiter':500,'ftol':1e-12,'gtol':1e-8})
            if res.fun < best_nll:
                best_nll = res.fun; best_x = res.x; converged = res.success
        except: continue

    if best_x is None:
        best_x = np.zeros(k); best_x[:p] = yw
        best_nll = _css_nll(best_x, y, p, q); converged = False
        warnings.warn(f"ARMA({p},{q}): tất cả restarts thất bại, "
                      "dùng Yule-Walker fallback.", RuntimeWarning)

    ar = best_x[:p]; ma = best_x[p:]
    eu = compute_residuals(y,ar,ma)[p+q:]
    s2 = max(np.dot(eu,eu)/len(eu), 1e-10)
    return ar, ma, s2, 2*(k+1)+2*best_nll, converged


In [14]:
# ═══════════════════════════════════════════════════════════════════════════
# 6. GRID SEARCH — ADF-guided, p ≤ 3, q ≤ 3
# ═══════════════════════════════════════════════════════════════════════════
#
# d: chọn riêng bằng ADF test (không grid search d)
# p,q: grid search → AIC

def search_order(y_train, p_max=3, d_max=2, q_max=3,
                 fit_n=3000, n_restarts=8, alpha=0.05, verbose=True):
    """
    Grid search ARIMA(p,d,q).
    fit_n=3000: fit trên 3000 điểm cuối train —
    params converge với n lớn nên đủ chính xác, nhanh hơn ~10x.
    """
    if verbose:
        print(f"  p∈{{0..{p_max}}}, q∈{{0..{q_max}}}, d tự động bằng ADF")

    d = suggest_d(y_train, d_max=d_max, alpha=alpha, verbose=verbose)
    y_diff = difference(y_train, d)
    y_fit  = y_diff[-fit_n:] if len(y_diff) > fit_n else y_diff

    best_aic, best = np.inf, None
    for p in range(p_max+1):
        for q in range(q_max+1):
            if p == 0 and q == 0: continue
            try:
                ar,ma,s2,aic,conv = fit_arma(y_fit, p, q, n_restarts)
                warn = "" if conv else " ⚠"
                if verbose: print(f"    ARIMA({p},{d},{q}) AIC={aic:9.2f}{warn}")
                if aic < best_aic:
                    best_aic = aic
                    best = {'order':(p,d,q), 'ar':ar, 'ma':ma, 'sigma2':s2}
            except Exception as e:
                if verbose: print(f"    ARIMA({p},{d},{q}) failed: {e}")

    if verbose:
        print(f"  → Best: ARIMA{best['order']}  AIC={best_aic:.2f}")
    return best['order'], best['ar'], best['ma'], best['sigma2'], best_aic


In [15]:
# ═══════════════════════════════════════════════════════════════════════════
# 7. ROLLING ONE-STEP FORECAST
# ═══════════════════════════════════════════════════════════════════════════
#
# Tại mỗi bước t:
#   ŷ_t = φ1·y_{t-1} + ··· + θ1·ε_{t-1} + ···
#   ε_t = y_t − ŷ_t   (append vào history)
# Không refit params — đúng chuẩn evaluation pred_len=1.

def rolling_forecast(y_train, y_test, order, ar, ma):
    p, d, q = order
    y_tr_d  = difference(y_train, d)
    eps_tr  = compute_residuals(y_tr_d, ar, ma)
    y_hist  = list(y_tr_d)
    eps_hist = list(eps_tr)

    seed       = y_train[-(d+1):] if d > 0 else y_train[-1:]
    y_te_diff  = difference(np.concatenate([seed, y_test]), d)[:len(y_test)]

    preds_diff = []
    for t in range(len(y_te_diff)):
        pred  = sum(ar[i]*y_hist[-(i+1)]   for i in range(p))
        pred += sum(ma[j]*eps_hist[-(j+1)] for j in range(q))
        preds_diff.append(pred)
        eps_hist.append(y_te_diff[t] - pred)
        y_hist.append(y_te_diff[t])

    preds_diff = np.array(preds_diff)
    return inverse_difference(preds_diff, y_train, d) if d > 0 else preds_diff


In [16]:
# ═══════════════════════════════════════════════════════════════════════════
# 8. ARIMAForecaster — wrapper
# ═══════════════════════════════════════════════════════════════════════════

class ARIMAForecaster:
    """
    Full pipeline:
      eda()            → ADF + ACF/PACF plots (optional, trước search)
      search_orders()  → grid search (p,d,q) per target
      forecast()       → rolling one-step forecast trên test set
      diagnostics()    → Ljung-Box residual check
      evaluate()       → metrics dùng compute_metrics() từ notebook
    """

    def __init__(self, dataset, p_max=3, d_max=0, q_max=3,
                 fit_n=500, n_restarts=8, alpha=0.05, verbose=True):
        self.dataset = dataset
        self.p_max=p_max; self.d_max=d_max; self.q_max=q_max
        self.fit_n=fit_n; self.n_restarts=n_restarts
        self.alpha=alpha; self.verbose=verbose
        self.orders={}; self.ar={}; self.ma={}; self.sigma2={}
        self._results=None

    def eda(self, n_lags=40):
        """ADF test + ACF/PACF plots cho mỗi target. Chạy trước search_orders()."""
        print("="*60); print("EDA — Stationarity + ACF/PACF"); print("="*60)
        for col in self.dataset.target_cols:
            print(f"\n[{col}]")
            mu = self.dataset.scaler_stats[col]["mean"] if self.dataset.normalize else 0.
            sd = self.dataset.scaler_stats[col]["std"]  if self.dataset.normalize else 1.
            y = self.dataset.y_train[col].values.astype(np.float64) * sd + mu
            d = suggest_d(y, d_max=self.d_max, alpha=self.alpha, verbose=True)
            plot_acf_pacf(y, col_name=col, n_lags=n_lags, d=d if d>0 else None)

    def search_orders(self):
        print("="*60)
        print("GRID SEARCH — ARIMA(p,d,q) per target")
        print("="*60)
        for col in self.dataset.target_cols:
            print(f"\n[{col}]")
            mu = self.dataset.scaler_stats[col]["mean"] if self.dataset.normalize else 0.
            sd = self.dataset.scaler_stats[col]["std"]  if self.dataset.normalize else 1.
            y = self.dataset.y_train[col].values.astype(np.float64) * sd + mu
            order,ar,ma,s2,_ = search_order(
                y, self.p_max, self.d_max, self.q_max,
                self.fit_n, self.n_restarts, self.alpha, self.verbose)
            self.orders[col]=order; self.ar[col]=ar
            self.ma[col]=ma;        self.sigma2[col]=s2
        print("\n" + "="*60)
        print("Summary:")
        for col in self.dataset.target_cols:
            print(f"  {col:>12s}: ARIMA{self.orders[col]}")
        print("="*60)

    def forecast(self, split="test"):
        assert self.orders, "Chạy search_orders() trước."
        y_split = self.dataset.y_test if split=="test" else self.dataset.y_val
        results = {}
        print(f"\nRolling forecast — {split} set...")
        for col in self.dataset.target_cols:
            mu = self.dataset.scaler_stats[col]["mean"] if self.dataset.normalize else 0.
            sd = self.dataset.scaler_stats[col]["std"]  if self.dataset.normalize else 1.
            # Feed ARIMA raw data (trước normalize) để ADF/fitting đúng
            y_tr = self.dataset.y_train[col].values.astype(np.float64) * sd + mu
            y_te = y_split[col].values.astype(np.float64)              * sd + mu
            preds_s = rolling_forecast(y_tr, y_te, self.orders[col], self.ar[col], self.ma[col])
            # Predictions đã ở raw scale, không cần inverse transform thêm
            y_pred_r = preds_s; y_true_r = y_te
            results[col] = {"y_true":y_true_r, "y_pred":y_pred_r,
                            "residuals":y_true_r-y_pred_r}
            print(f"  [{col}] {len(y_te)} steps.")
        self._results = results
        return results

    def diagnostics(self, lags=[5,10,20]):
        """Ljung-Box trên forecast residuals. p > 0.05 = white noise = OK."""
        assert self._results, "Chạy forecast() trước."
        print("\nResidual Diagnostics (Ljung-Box) — H0: white noise")
        for col, res in self._results.items():
            print_ljung_box(ljung_box(res["residuals"], lags=lags), col)

    def evaluate(self, results=None, verbose=True):
        if results is None: results = self._results
        assert results, "Chạy forecast() trước."
        per_target = {}; all_true=[]; all_pred=[]
        for col, res in results.items():
            m = compute_metrics(res["y_true"], res["y_pred"])
            per_target[col] = m
            all_true.append(res["y_true"]); all_pred.append(res["y_pred"])
        overall = compute_metrics(np.concatenate(all_true), np.concatenate(all_pred))
        if verbose:
            print("\n=== ARIMA — TEST SET ===")
            for col,m in per_target.items():
                print(f"  {col:>12s} | {_format_metrics(m)}")
            print("-"*80)
            print(f"  {'OVERALL':>12s} | {_format_metrics(overall)}")
            print("="*80)
        return {"overall":overall, "per_target":per_target}


In [17]:
# ═══════════════════════════════════════════════════════════════════════════
# 9. RUN
# ═══════════════════════════════════════════════════════════════════════════
import time

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

dataset = TimeSeriesDataset(
    name="beijing_air_quality",
    seq_len=24, pred_len=1,
    train_ratio=0.7, val_ratio=0.2, test_ratio=0.1,
    batch_size=64, normalize=True,
)


# ── Fit ───────────────────────────────────────────────────────────────────────
arima = ARIMAForecaster(
    dataset     = dataset,
    p_max       = 3,       # p ∈ {0..3}
    d_max       = 2,       # d tự động bằng ADF
    q_max       = 3,       # q ∈ {0..3}
    fit_n       = 3000,    # fit trên 3000 điểm cuối train (~7s/model)
    n_restarts  = 8,
    alpha       = 0.05,
    verbose     = True,
)

t0 = time.time()
arima.search_orders()
print(f"\nGrid search: {(time.time()-t0)/60:.1f} phút")

# ── Forecast + Diagnostics ────────────────────────────────────────────────────
arima_results = arima.forecast(split="test")
arima.diagnostics(lags=[5, 10, 20])

GRID SEARCH — ARIMA(p,d,q) per target

[PM2.5]
  p∈{0..3}, q∈{0..3}, d tự động bằng ADF
  ADF stationarity test:
    d=0: ADF=-16.484  p=0.0010  crit(5%)=-2.862  ✓ stationary
    ARIMA(0,0,1) AIC= 33909.10
    ARIMA(0,0,2) AIC= 32123.91
    ARIMA(0,0,3) AIC= 30972.54
    ARIMA(1,0,0) AIC= 27139.04
    ARIMA(1,0,1) AIC= 26965.11
    ARIMA(1,0,2) AIC= 26913.36
    ARIMA(1,0,3) AIC= 26906.01
    ARIMA(2,0,0) AIC= 27123.13
    ARIMA(2,0,1) AIC= 26912.40
    ARIMA(2,0,2) AIC= 26905.25
    ARIMA(2,0,3) AIC= 26898.07
    ARIMA(3,0,0) AIC= 27082.29
    ARIMA(3,0,1) AIC= 26905.48
    ARIMA(3,0,2) AIC= 26898.35
    ARIMA(3,0,3) AIC= 26890.52
  → Best: ARIMA(3, 0, 3)  AIC=26890.52

[PM10]
  p∈{0..3}, q∈{0..3}, d tự động bằng ADF
  ADF stationarity test:
    d=0: ADF=-16.011  p=0.0010  crit(5%)=-2.862  ✓ stationary
    ARIMA(0,0,1) AIC= 34820.76
    ARIMA(0,0,2) AIC= 32797.08
    ARIMA(0,0,3) AIC= 31717.90
    ARIMA(1,0,0) AIC= 28324.83
    ARIMA(1,0,1) AIC= 28317.37
    ARIMA(1,0,2) AIC= 28307.62

In [18]:
# ── Evaluate ──────────────────────────────────────────────────────────────────
_, _, test_loader = dataset.get_loaders()
baseline_metrics = evaluate_naive_baseline(dataset, test_loader, verbose=True)
arima_metrics    = arima.evaluate()

# So sánh ARIMA vs Naive Baseline 
keys = ["MAE", "MSE", "RMSE", "WMAPE", "R2"]
b    = baseline_metrics["overall"]
a    = arima_metrics["overall"]
print("\n=== SO SÁNH: ARIMA vs Naive Baseline ===")
print(f"  {'Metric':>8}  {'Naive':>10}  {'ARIMA':>10}  {'Better?':>10}")
print("  " + "-"*44)
for k in keys:
    better = ("ARIMA ✓" if (k!="R2" and a[k]<b[k]) or (k=="R2" and a[k]>b[k])
              else "Naive")
    print(f"  {k:>8}  {b[k]:>10.4f}  {a[k]:>10.4f}  {better:>10}")
print("="*50)



=== NAIVE BASELINE ===
     PM2.5 | MSE: 560.0901 | MAE: 12.4416 | RMSE: 23.6662 | WMAPE: 0.1228 | R2: 0.9499
      PM10 | MSE: 857.7209 | MAE: 16.7040 | RMSE: 29.2869 | WMAPE: 0.1425 | R2: 0.9291
       SO2 | MSE: 28.3353 | MAE: 2.7752 | RMSE: 5.3231 | WMAPE: 0.1858 | R2: 0.8969
       NO2 | MSE: 231.6196 | MAE: 8.4844 | RMSE: 15.2191 | WMAPE: 0.1182 | R2: 0.8724
        CO | MSE: 370941.7188 | MAE: 288.5731 | RMSE: 609.0498 | WMAPE: 0.1591 | R2: 0.8776
        O3 | MSE: 135.7468 | MAE: 6.3250 | RMSE: 11.6510 | WMAPE: 0.2106 | R2: 0.8545
--------------------------------------------------------------------------------
OVERALL    | MSE: 62125.8750 | MAE: 55.8839 | RMSE: 249.2506 | WMAPE: 0.1560 | R2: 0.9335

=== ARIMA — TEST SET ===
         PM2.5 | MSE: 1398.2518 | MAE: 20.5063 | RMSE: 37.3932 | WMAPE: 0.2028 | R2: 0.8741
          PM10 | MSE: 1950.9957 | MAE: 25.7996 | RMSE: 44.1701 | WMAPE: 0.2204 | R2: 0.8378
           SO2 | MSE: 65.2053 | MAE: 4.3920 | RMSE: 8.0750 | WMAPE: 0.2958